In [1]:
import os
import sys
import numpy as np
import cv2
from dataengine.generator.obstacle.debug_utils import draw_mask, Statistics

# unified_data_path = '/home/mars/mars_test'
data_path = './'

unified_file_name = '2023_10_19_07_21_48_seg_53.npy'
unified_data_path = os.path.join(data_path, unified_file_name)

tracklet_file_name = 'tracklet.npy'
tracklet_file_path = os.path.join(data_path, tracklet_file_name)

unified_npy = np.load(unified_data_path)
tracklet_npy = np.load(tracklet_file_path)

In [2]:
# for i in range(tracklet_npy.shape[0]):
#     ns = tracklet_npy['timestamp'][i]
#     idx = tracklet_npy['idx'][i]
#     K = tracklet_npy['K'][i]
#     D = tracklet_npy['D'][i]
#     img = tracklet_npy['image'][i]
#     mask = tracklet_npy['mask'][i]

#     print(np.sum(mask))

#     render = img.copy()
#     draw_mask(render, mask)
#     cv2.imshow("", render)
#     key = cv2.waitKey(0)
#     if key == 27:
#         break
# cv2.destroyAllWindows()

In [3]:
from marsdataio.npyhelper import get_all_cameras_params
from marsdataio.dbhelper import (
    extract_image,
    collate_images,
)

timestamps = np.load('saved_ns.npy')

sensors = unified_npy['sensors']
cam_params = list(get_all_cameras_params(sensors).values())

time_to_images = {}

cap = cv2.VideoCapture('origin.mp4')
video_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
video_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

ret, dst = cap.read()

count = 0

while ret:
    imgs = []
    for cam_param in cam_params:
        img = extract_image(dst, cam_param.img_wh, cam_param.video_pos)
        imgs.append(img)

    time_to_images[timestamps[count]] = imgs

    count += 1
    ret, dst = cap.read()

kj/filesystem-disk-unix.c++:1690: warning: PWD environment variable doesn't match current directory; pwd = /home/mars


In [ ]:
from marstransform.lietrans import quat_rotate_np, make_se3_np, inv_se3_np
from marstransform.loctrans import sensor_to_body
from marsdataio.npyhelper import get_imu_params
import rerun as rr  # pip install rerun-sdk
import rerun.blueprint as rrb
import argparse

EGO_KEY = 'world/ego'

my_blueprint = rrb.Blueprint(
    rrb.Vertical(
        rrb.Spatial3DView(name='3D', origin='world'),
        rrb.Spatial3DView(
            name="3D2",
            origin=EGO_KEY,
            contents=['world/ego/**'],
        ),
        # rrb.Vertical(
        #     rrb.Spatial2DView(
        #         name="BGR",
        #         origin=EGO_KEY,
        #         contents=[f'{CAMERA_KEY}/bgr', f'{CAMERA_KEY}/mask'],
        #     ),
        #     rrb.Spatial3DView(
        #         name="mask", origin=EGO_KEY, contents=['$origin/**']
        #     ),
        # ),
    ),
)

rr.init("hi", spawn=True)
rr.send_blueprint(my_blueprint)

ego_poses = unified_npy['frames_data']['ecef_pos']
ego_quats = unified_npy['frames_data']['ecef_quat']

_, imu_params = get_imu_params(sensors)
_imu_extrinsic = imu_params.T_mi


debug_cam_idxs = [0, 1, 3, 4, 5, 6]

T_bcs = []
for cam_param in cam_params:
    T_bc = sensor_to_body(_imu_extrinsic, cam_param.T_mc)
    T_bcs.append(T_bc)

w, h = cam_param.img_wh

T_ecs = {}


rr.log(
    "world/origin",
    rr.Points3D([0.0, 0.0, 0.0], radii=0.50, colors=[255, 200, 10]),
    timeless=True,
)

start_from = 160
dura = 20
debug_timestamps = []
b_pos = ego_poses[start_from]
ego_poses = ego_poses - b_pos


for i, (ego_pos, ego_quat) in enumerate(zip(ego_poses, ego_quats)):
    # if i > 3:
    #     break
    if i < start_from:
        continue

    rr.set_time_nanos("time", timestamps[i])
    debug_timestamps.append(timestamps[i])

    paths = ego_poses[start_from : i + 1]
    rr.log('world/path', rr.LineStrips3D(paths))

    pos = ego_poses[i]

    T_ego_eb = make_se3_np(quat_rotate_np(ego_quat), ego_pos)
    rr.log(
        EGO_KEY,
        rr.Transform3D(translation=T_ego_eb[:3, 3], mat3x3=T_ego_eb[:3, :3]),
    )

    for idx in debug_cam_idxs:
        cam_param = cam_params[idx]
        T_bc = T_bcs[idx]
        T_ec = T_ego_eb @ T_bc

        idx_to_Tec = {}
        idx_to_Tec[idx] = T_ec

        T_ecs[timestamps[i]] = idx_to_Tec

        rr.log(
            f'world/ego/cam{idx}',
            rr.Transform3D(translation=T_bc[:3, 3], mat3x3=T_bc[:3, :3]),
        )
        dist = 20
        if idx == 0:
            dist = 30
        rr.log(
            f'world/ego/cam{idx}',
            rr.Pinhole(
                image_from_camera=cam_param.K,
                resolution=[w, h],
                image_plane_distance=dist,
            ),
        )
        # continue
        bgr = time_to_images[timestamps[i]][idx]
        rr.log(
            f'world/ego/cam{idx}/bgr',
            rr.Image(bgr, color_model="BGR").compress(jpeg_quality=95),
        )


for i in range(tracklet_npy.shape[0]):
    idx = tracklet_npy['idx'][i]
    if idx not in debug_cam_idxs:
        continue

    time = tracklet_npy['timestamp'][i]
    if time < timestamps[start_from]:
        continue

    rr.set_time_nanos("time", time)
    rr.log("/", rr.AnnotationContext([(0, "none", (0, 0, 0))]), static=True)
    rr.log(
        f'world/ego/cam{idx}/bgr/mask',
        rr.SegmentationImage(tracklet_npy['mask'][i]),
    )

[2024-11-15T03:31:01Z INFO  re_sdk::spawn] A process is already listening at this address. Assuming it's a Rerun Viewer. addr=0.0.0.0:9876
[2024-11-15T03:31:01Z INFO  re_sdk_comms::server] New SDK client connected from: 127.0.0.1:39164


In [ ]:
from marsneuralzoo.models.e2emvm import E2emvm

e2emvm = E2emvm()

# prev0 -curr0
# prev1 _ curr1

for time in debug_timestamps:
    # preds = []
    # 0-1
    images = []
    for idx in debug_cam_idxs:
        images.append(time_to_images[time][idx])

    torch_images, features = e2emvm.extract_features(images)
    preds = e2emvm(torch_images, features)

Loaded SuperPoint model


In [11]:
mask_times = []
for i in range(tracklet_npy.shape[0]):
    cam_idx = tracklet_npy['idx'][i]
    if cam_idx != debug_cam_idx:
        continue

    time = tracklet_npy['timestamp'][i]
    mask_times.append(time)


target_idx = 3
target_mask = tracklet_npy['mask'][target_idx]


duration = 5
target_imgs = []
for i in range(duration):
    target_time = mask_times[target_idx + i]
    target = time_to_images[target_time][debug_cam_idx].copy()
    target = cv2.cvtColor(target, cv2.COLOR_BGR2GRAY)
    target_imgs.append(target)


import cv2

fast = cv2.FastFeatureDetector_create()
kpts = fast.detect(target_imgs[0], None)
pts = []

for kpt in kpts:
    pts.append(kpt.pt)

# debug_img = cv2.drawKeypoints(debug_img, kpts, None, color=(255, 0, 0))

# import matplotlib.pyplot as plt
# plt.imshow(debug_img)

In [12]:
pts_in_mask = []
for pt in pts:
    ix = int(pt[0])
    iy = int(pt[1])

    if target_mask[iy, ix] == 1:
        pts_in_mask.append(pt)

tracked_pts = []

valid_idx = np.full((len(pts_in_mask)), True, dtype=bool)

prev_pts = np.array(pts_in_mask, dtype=np.float32)
tracked_pts.append(prev_pts)

for i in range(duration - 1):

    curr_pts, status, err = cv2.calcOpticalFlowPyrLK(
        target_imgs[i], target_imgs[i + 1], prev_pts, None
    )

    reversed_pts, status, err = cv2.calcOpticalFlowPyrLK(
        target_imgs[i + 1], target_imgs[i], curr_pts, None
    )

    diff = np.linalg.norm(prev_pts - reversed_pts, axis=1)

    valid_idx = valid_idx & (diff < 0.04)
    tracked_pts.append(curr_pts)
    prev_pts = curr_pts

# for i, pts in enumerate(tracked_pts):
#     pts = pts[valid_idx]
#     target_image = targets[i].copy()
#     render = cv2.cvtColor(target_image, cv2.COLOR_GRAY2BGR)
#     for pt in pts:
#         cv2.circle(render,(int(pt[0]),int(pt[1])), 3,(0,255,0))
#     cv2.imshow("",render)
#     cv2.waitKey(0)

# cv2.destroyAllWindows()

In [13]:
tracked_undists = []
undistorted_imgs = []


size = (target_imgs[0].shape[1], target_imgs[0].shape[0])

map1, map2 = cv2.fisheye.initUndistortRectifyMap(
    cam_param.K,
    cam_param.distort_coefs,
    np.eye(3),
    cam_param.K,
    size,
    cv2.CV_16SC2,
)

for img, pts in zip(target_imgs, tracked_pts):
    pts = pts.reshape(-1, 1, 2)
    undists = cv2.fisheye.undistortPoints(
        pts,
        cam_param.K,
        cam_param.distort_coefs,
        R=np.eye(3, dtype=np.float32),
        P=np.eye(3, dtype=np.float32),
    )

    ones_array = np.ones(
        (undists.shape[0], undists.shape[1], 1), dtype=undists.dtype
    )
    undists = np.concatenate((undists, ones_array), axis=2)

    undist_img = cv2.remap(
        img,
        map1,
        map2,
        interpolation=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
    )
    tracked_undists.append(undists)
    undistorted_imgs.append(undist_img)

In [ ]:
target_times = []
for i in range(duration):
    target_time = mask_times[target_idx + i]
    target_times.append(target_time)
    undist_img = undistorted_imgs[i]
    rr.set_time_nanos('time', target_time)
    # print(target_time)

    cv2.imwrite('./sample.png', undist_img)

    rr.log(
        f'{CAMERA_KEY}/gray',
        rr.Image(undist_img).compress(jpeg_quality=95),
    )

In [ ]:
def x_is_Gz_plus_H_get_G_H(T, u, v):
    r00, r01, r02, r10, r11, r12, r20, r21, r22 = T[:3, :3].flatten()
    px, py, pz = T[:3, 3]

    """
    u = X' / Z'
    v = Y' / Z'
    
    [X' Y' Z']^T = R * [X Y Z]^T + P
    
    1... Y = AX + BZ + C
    2... X = DY + EZ + F   
    3... X = G * Z + H
    """

    e0 = v * r21 - r11
    e1 = r10 - v * r20
    e2 = r12 - v * r22
    e3 = py - v * pz

    A = e1 / e0
    B = e2 / e0
    C = e3 / e0

    e4 = u * r20 - r00
    e5 = r01 - u * r21
    e6 = r02 - u * r22
    e7 = px - u * pz

    D = e5 / e4
    E = e6 / e4
    F = e7 / e4

    G = (B * D + E) / (1 - A * D)
    H = (C * D + F) / (1 - A * D)

    return G, H


# print(tracked_undists)

for i in range(1, duration - 1):

    T_ec0 = T_ecs[target_times[i - 1]]
    T_ec1 = T_ecs[target_times[i]]
    # T_c1e = inv_se3_np(T_ec1)
    T_ec2 = T_ecs[target_times[i + 1]]

    T_c0c1 = inv_se3_np(T_ec0) @ T_ec1
    T_c2c1 = inv_se3_np(T_ec2) @ T_ec1

    print(T_ec0[:3, 3])

    u0, v0, _ = tracked_undists[i - 1][0][0]
    u1, v1, _ = tracked_undists[i][0][0]
    u2, v2, _ = tracked_undists[i + 1][0][0]

    G, H = x_is_Gz_plus_H_get_G_H(T_c0c1, u0, v0)
    I, K = x_is_Gz_plus_H_get_G_H(T_c2c1, u2, v2)

    z0 = (2 * u1 - H - K - 2 * I) / (G - I)
    z2 = (2 * u1 - H - K - 2 * G) / (I - G)
    print(z0, z2)

[ 91.74775083 -65.19977572 141.23559456]
0.16229965921555625 1.8377003407844437
[ 92.14356026 -65.47792107 141.84465873]
-2.03187891199365 4.03187891199365
[ 92.56323663 -65.77282181 142.49213875]
1.3339523607439618 0.6660476392560382


In [ ]:
def x_is_Gz_plus_H_get_G_H(T, u, v):
    r00, r01, r02, r10, r11, r12, r20, r21, r22 = T[:3, :3].flatten()
    px, py, pz = T[:3, 3]

    """
    u = X' / Z'
    v = Y' / Z'
    
    [X' Y' Z']^T = R * [X Y Z]^T + P
    
    1... Y = AX + BZ + C
    2... X = DY + EZ + F   
    3... X = G * Z + H
    """

    e0 = v * r21 - r11
    e1 = r10 - v * r20
    e2 = r12 - v * r22
    e3 = py - v * pz

    A = e1 / e0
    B = e2 / e0
    C = e3 / e0

    e4 = u * r20 - r00
    e5 = r01 - u * r21
    e6 = r02 - u * r22
    e7 = px - u * pz

    D = e5 / e4
    E = e6 / e4
    F = e7 / e4

    G = (B * D + E) / (1 - A * D)
    H = (C * D + F) / (1 - A * D)

    return G, H


u0 = 0.5
v0 = 0
u1 = 0.5
v1 = 0
u2 = 0.5
v2 = 0

Twc0 = np.eye(4, 4)

Twc1 = np.eye(4, 4)
Twc1[0, 3] = 1

Twc2 = np.eye(4, 4)
Twc2[0, 3] = 2


T_c0c1 = inv_se3_np(Twc0) @ Twc1
T_c2c1 = inv_se3_np(Twc2) @ Twc1

print(T_c2c1)

G, H = x_is_Gz_plus_H_get_G_H(T_c0c1, u0, v0)
I, K = x_is_Gz_plus_H_get_G_H(T_c2c1, u2, v2)
print(G, H, I, K)
z0 = (2 * u1 - H - K - 2 * I) / (G - I + 0.000000000000001)
z2 = (2 * u1 - H - K - 2 * G) / (I - G + 0.000000000000001)

print(z0, z2)

[[ 1.  0.  0. -1.]
 [ 0.  1.  0.  0.]
 [ 0.  0.  1.  0.]
 [ 0.  0.  0.  1.]]
0.5 -1.0 0.5 1.0
0.0 0.0
